# AI Wine Sommelier RAG

## Wine Review Indexing

https://www.kaggle.com/datasets/christopheiv/winemagdata130k

In [1]:
%pip install -Uqqq langchain langchain-community langchain-openai langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

## Pinecone 테스트

In [3]:
# 데이터로드
from langchain_core.documents import Document

documents = [
    Document(page_content="LangChain은 LLM 기반 애플리케이션을 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://langchain.com/docs", "author": "alice", "page": 1}),
    Document(page_content="ChromaDB는 오픈소스 벡터 데이터베이스입니다.", metadata={"source": "https://chromadb.org/intro", "license": "MIT", "date": "2024-07-01"}),
    Document(page_content="파이썬으로 AI 서비스를 개발할 수 있습니다.", metadata={"source": "https://pythonai.co.kr", "editor": "kim", "page": 7}),
    Document(page_content="LLM은 자연어 처리를 위한 대형 언어 모델을 의미합니다.", metadata={"source": "https://llmwiki.com/info", "author": "bob", "version": "v1.1"}),
    Document(page_content="RAG는 검색과 생성의 결합 방식을 제공합니다.", metadata={"source": "https://rag-search.io", "reviewer": "lee", "section": "summary"}),
    Document(page_content="벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.", metadata={"source": "https://vectorbase.net", "author": "jin", "topic": "vector"}),
    Document(page_content="LangChain을 이용하면 다양한 AI 파이프라인을 구축할 수 있습니다.", metadata={"source": "https://langchain.com/blog", "editor": "sarah", "date": "2024-06-30"}),
    Document(page_content="OpenAI의 GPT 모델은 텍스트 생성에 특화되어 있습니다.", metadata={"source": "https://openai.com/gpt", "lang": "ko", "page": 5}),
    Document(page_content="파이썬은 AI 및 데이터 분석 분야에서 널리 사용되는 언어입니다.", metadata={"source": "https://python.org/usecases", "author": "chun", "updated": "2024-05"}),
    Document(page_content="Streamlit은 파이썬으로 대시보드를 쉽게 만들 수 있는 프레임워크입니다.", metadata={"source": "https://streamlit.io/start", "editor": "park", "date": "2024-04-28"}),
    Document(page_content="Dense Retrieval은 임베딩 벡터를 이용한 검색 방식을 의미합니다.", metadata={"source": "https://retrieval.ai/dense", "type": "tech", "page": 3}),
    Document(page_content="Pandas 라이브러리는 데이터 분석에 자주 사용됩니다.", metadata={"source": "https://pandas.pydata.org/about", "maintainer": "koh", "section": "intro"}),
    Document(page_content="메타데이터 필터링은 검색 결과의 품질을 높여줍니다.", metadata={"source": "https://search.com/metadata", "author": "seo", "feature": "filter"}),
    Document(page_content="SelfQueryRetriever는 자연어 쿼리를 임베딩 쿼리로 변환해줍니다.", metadata={"source": "https://selfquery.ai", "editor": "min", "date": "2024-05-12"}),
    Document(page_content="프롬프트 엔지니어링은 LLM의 성능을 극대화하는 방법입니다.", metadata={"source": "https://prompting.dev/guide", "author": "yang", "topic": "prompt"}),
    Document(page_content="HyDE 기법은 하이브리드 검색에 사용됩니다.", metadata={"source": "https://hyde-tech.com", "reviewer": "kang", "version": "2024.1"}),
    Document(page_content="CoT는 복잡한 문제를 단계적으로 해결하는 프롬프트 기법입니다.", metadata={"source": "https://cotprompt.org", "editor": "jung", "date": "2023-12-01"}),
    Document(page_content="문서 임베딩은 텍스트를 고차원 벡터로 변환하는 과정입니다.", metadata={"source": "https://embedding.ai/intro", "section": "embedding", "author": "song"}),
    Document(page_content="CrewAI는 멀티 에이전트 시스템 구현을 돕는 툴입니다.", metadata={"source": "https://crew.ai/docs", "lang": "ko", "page": 9}),
    Document(page_content="Fine-tuning은 사전학습 모델을 특정 도메인에 맞게 재학습시키는 과정입니다.", metadata={"source": "https://finetune.ai/guide", "editor": "jeon", "date": "2024-01-30"})
]

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

# 문서 -> 임베딩 -> Pinecore 업로드
vector_store = PineconeVectorStore.from_documents(
    documents,
    embeddings,
    index_name = 'pinecone-test'
)

In [5]:
retriever = vector_store.as_retriever(
    search_typ = 'similarity',
    search_kwargs = {'k':5}
)

retriever.invoke('벡터 데이터베이스란?')

[Document(id='e281be61-5665-4785-bc1f-ac675a2531ea', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='38461e00-d5fe-4e75-b3c9-ef1400ae06fe', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='ab7e4776-661b-4973-8f9b-bbc8bc7e3496', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='2897c764-7196-40f7-8366-e19f667c8409', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.'),
 Document(id='cd6e3f32-2607-4a05-975c-fbad4978ca3e', metadata={'author': 'jin', 'source': 'https://vectorbase.net', 'topic': 'vector'}, page_content='벡터 데이터베이스는 임베딩된 데이터를 효율적으로 검색할 수 있습니다.')]

In [6]:
!gdown 1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3

Downloading...
From: https://drive.google.com/uc?id=1sJzwWez1Q7hZqlWOV70Ng00_taDqYPi3
To: c:\Users\UK\SKN\LLM\06_2stage_rag\winemag-data-130k-v2.csv

  0%|          | 0.00/52.9M [00:00<?, ?B/s]
  1%|          | 524k/52.9M [00:00<00:13, 3.98MB/s]
  7%|▋         | 3.67M/52.9M [00:00<00:02, 17.4MB/s]
 14%|█▍        | 7.34M/52.9M [00:00<00:02, 21.0MB/s]
 24%|██▍       | 12.6M/52.9M [00:00<00:01, 30.8MB/s]
 35%|███▍      | 18.4M/52.9M [00:00<00:00, 38.5MB/s]
 44%|████▎     | 23.1M/52.9M [00:00<00:00, 40.7MB/s]
 53%|█████▎    | 27.8M/52.9M [00:00<00:00, 41.5MB/s]
 62%|██████▏   | 33.0M/52.9M [00:00<00:00, 43.4MB/s]
 71%|███████▏  | 37.7M/52.9M [00:01<00:00, 39.2MB/s]
 79%|███████▉  | 41.9M/52.9M [00:01<00:00, 39.4MB/s]
 88%|████████▊ | 46.7M/52.9M [00:01<00:00, 40.9MB/s]
 97%|█████████▋| 51.4M/52.9M [00:01<00:00, 42.2MB/s]
100%|██████████| 52.9M/52.9M [00:01<00:00, 37.2MB/s]


In [7]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader('winemag-data-130k-v2.csv',encoding='utf-8')
docs = loader.load()
print(len(docs))

C:\Users\UK\AppData\Local\Temp\ipykernel_51832\4027444281.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


129971


In [8]:
for i,doc in enumerate(docs[:2]):
    print(f"{i} : {type(doc)}")
    print(f"{doc.metadata}")
    print(f"{doc.page_content}")
    print()

0 : <class 'langchain_core.documents.base.Document'>
{'source': 'winemag-data-130k-v2.csv', 'row': 0}
: 0
country: Italy
description: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
designation: Vulkà Bianco
points: 87
price: 
province: Sicily & Sardinia
region_1: Etna
region_2: 
taster_name: Kerin O’Keefe
taster_twitter_handle: @kerinokeefe
title: Nicosia 2013 Vulkà Bianco  (Etna)
variety: White Blend
winery: Nicosia

1 : <class 'langchain_core.documents.base.Document'>
{'source': 'winemag-data-130k-v2.csv', 'row': 1}
: 1
country: Portugal
description: This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red berry fruits and freshened with acidity. It's  already drinkable, although it will certainly be better from 2016.
designation: Avidagos
points: 87
price: 15.0
province: Douro
region_1: 
region_2: 
tast

In [9]:
# Pinecone 업로드 : Pinecone 인덱스에 연결된 벡터스토어 객체 생성   
vector_store = PineconeVectorStore(
    index_name= 'winemag-data',
    embedding= embeddings
)

batch_size = 100

for i in range(0, len(docs),batch_size):    # 전체 docs를 batch_size 단위로 순회
    batch_data = docs[i:i+batch_size]       # 해당 인덱스 + 100 씩 리스트가져옴
    vector_store.add_documents(batch_data)  # 배치 Document들을 임베딩 -> Pinecone 업로드
    print(f"index:{i} ~ {i+batch_size}")

KeyboardInterrupt: 

## Retrieval & Generation
1. 텍스트/이미지 입력으로 요리에 설명 chain
2. 요리설명텍스트 벡터db조회 chain
3. 요리설명/리뷰검색을 가지고 와인추천 응답 chain

### 요리설명 chain

In [13]:
# 채팅 프롬프트 / 휴먼 메시지 템플릿

from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser   # 출력 -> 문자열 파싱
from langchain_core.runnables import RunnableLambda # 함수를 Runnable로 감싸 chain에서 실행

def describe_dish_flavor(query:dict):
    prompt = ChatPromptTemplate.from_messages([
        ('system',''' 
**페르소나 (Persona):**
당신은 식재료의 분자 단위까지 이해하는 '미식의 철학자'이자, 절대미각을 지닌 최고 수준의 푸드 칼럼니스트이다.
당신은 요리를 단순한 음식이 아닌, 식재료와 조리 과학(Culinary Science)이 빚어낸 예술 작품으로 바라본다.
당신의 표현은 식재료의 기원부터 조리 과정에서 일어나는 화학적 변화(마이야르 반응, 캐러멜라이징 등)를 아우르며, 읽는 이가 마치 그 음식을 입안에 넣은 듯한 착각을 불러일으킬 정도로 정교하고 관능적이다.

**역할 (Role):**
당신의 핵심 역할은 요리의 맛, 향, 텍스처(Texture), 그리고 밸런스를 해부학적으로 분석하여 전달하는 것이다.
1.  **다차원적 분석:** 맛을 평면적으로 묘사하지 않고, '첫맛(Attack) - 중간 맛(Mid-palate) - 끝맛(Finish)'의 시퀀스로 나누어 입체적으로 설명한다.
2.  **조리법과 맛의 인과관계:** 왜 이 맛이 나는지, 어떤 조리 테크닉이 식재료의 잠재력을 폭발시켰는지 논리적 근거를 제시한다.
3.  **미식의 가이드:** 식재료 간의 궁합(Pairing)과 풍미를 극대화하는 팁을 제공하여, 사용자의 미식 수준을 한 단계 끌어올린다.

**가이드라인 (Guidelines):**
- **감각의 구체화:** '맛있다', '부드럽다' 같은 추상적 표현을 금지한다. 대신 '혀를 감싸는 벨벳 같은 질감', '비강을 때리는 훈연 향' 등 구체적인 묘사를 사용하라.
- **단계별 서술:** 시각과 후각으로 시작해, 입안에서의 질감 변화, 그리고 목 넘김 후의 여운까지 단계별로 서술하라.

**예시 (Examples):**

* **사용자:** "잘 만든 '트러플 크림 리조또'의 맛을 묘사해 주세요."
    **당신:**
    * **[시각과 후각]** 김이 모락모락 나는 접시 위로 흙내음(Earthy)을 가득 머금은 트러플 향이 가장 먼저 코끝을 강타합니다. 크림소스의 녹진한 유분 향과 섞여 마치 가을 숲속에 와 있는 듯한 묵직한 아로마가 식욕을 자극합니다.
    * **[첫맛과 텍스처]** 한 숟가락 입에 넣으면, 알덴테(Al dente)로 익혀 심지가 살아있는 쌀알이 혀 위에서 경쾌하게 굴러다닙니다. 동시에 파르미지아노 레지아노 치즈가 녹아든 크림소스가 쌀알 사이사이를 끈적하게 메우며 혀를 포근하게 감싸 안습니다.
    * **[풍미의 폭발]** 씹을수록 버섯의 감칠맛(Umami)이 폭발합니다. 버터의 고소함이 베이스를 깔아주는 가운데, 트러플 오일의 강렬한 향이 비강으로 역류하며 미각을 지배합니다.
    * **[여운]** 목을 넘긴 후에도 트러플의 진한 향과 크림의 고소함이 입안에 길게 남아, 무거운 레드 와인 한 모금을 간절하게 부릅니다.

* **사용자:** "양파 수프(French Onion Soup)의 맛의 비결이 무엇인가요?"
    **당신:**
    * **[핵심 분석]** 이 요리의 영혼은 **'인내심이 만든 단맛'**에 있습니다. 양파를 약불에서 장시간 볶아내는 '캐러멜라이징(Caramelization)' 과정이 핵심입니다.
    * **[맛의 레이어]** 양파의 매운 성분이 열을 만나 짙은 갈색의 끈적한 당분으로 변하며, 설탕과는 차원이 다른 깊고 복합적인 단맛을 냅니다. 여기에 쇠고기 육수의 짭조름한 감칠맛이 더해져 '단짠'의 완벽한 균형을 이룹니다.
    * **[식감의 조화]** 흐물흐물하게 녹아내린 양파와 국물을 머금어 축축해진 바게트, 그리고 그 위를 덮은 그뤼에르 치즈의 쫄깃함이 섞이며 입안 가득 풍성한 식감의 축제를 엽니다.

**주의사항**
맛의 대한 묘사만 줄글 형식으로 50자이내로 작성하세요.
'''),
        ('human','사용자가 제공한 이미지의 요리명과 풍미를 잘 묘사해 주세요.')
    ])

    temp = []
    if query.get('image_urls'):
        temp +=[{"image_url" : image_url} for image_url in query.get('image_urls')]

    # 텍스트가있는경우 추라
    if query.get('text'):
        temp +=[{'text':query.get('text')}]

    # HumanMessagePromptTemplate : 멀티모달 블록형태의 값을 human 메시지로 프롬프트에 추가
    prompt += HumanMessagePromptTemplate.from_template(temp)

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain

# 입력을 받아 chain을 실행할 수 있는 Runnable
dish_flavor_chain = RunnableLambda(describe_dish_flavor)
response = dish_flavor_chain.invoke({
    "text":' ',
    'image_urls':[
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMjVfODIg%2FMDAxNzYxMzc2NjMzMDA4.-qxYpSDZfPleD8cj9VzxvqckYRIvaGpZW-fibT3whjsg.mefkB_k7NzVsXrb9RGPEQvZAplyzrustInLMV-827Gkg.JPEG%2FIMG%25A3%25DF2865.JPG&type=sc960_832',
        'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTA5MjBfMjEz%2FMDAxNzU4MzgwMDM4MzAz.XWaCn8Xu_7jjbYWq5P4MFqibaqNz4p3CFKRgjOnP2dMg.7wrfjF9U-p-CORf9ix4DbEGFRnOkaNh2ihjYlZOZy6Ag.JPEG%2FIMG_6083.JPG&type=sc960_832'
    ],
})

print(response)

치즈·등심·안심 돈가스와 매콤달콤한 제육볶음의 풍성한 한 상입니다.


## 리뷰 검색 chain

In [34]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

def search_wine_review(query):

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

    # PineconeVectorStore index 연결
    vector_store = PineconeVectorStore(
        index_name = 'winemag-data',    # 저장할 index명
        embedding= embeddings           # 질의문 임베딩에 사용할 모델
    )

    docs = vector_store.similarity_search(query, k=5)

    return {
        'dish_flavor' : query,
        'wine_reviews' : '\n\n'.join(doc.page_content for doc in docs)
    }

query = '''
첫 번째 이미지는 '허브 그릴 스테이크'입니다. 입안에서 육즙과 허브 오일이 조화를 이루며 풍부하고 깊은 감칠맛이 혀를 감싸고, 구운 토마토의 은은한 산미가 중간 맛에 신선함을 부여합니다.
두 번째 이미지는 '시저 샐러드'입니다. 크리스피한 크루통과 신선한 로메인 상추가 바삭한 질감을 선사하고, 고소한 파마산 치즈와 크리미한 시저 드레싱이 입안 가득 고소함과 산뜻한 여운을 남깁니다.
'''

print(search_wine_review(query)['wine_reviews'])

: 4988
country: US
description: Honey-sweet and direct, with citrus jam, apricot essence, crème brûlée and vanilla cream flavors. Fine with white cookies, lemon chiffon pie, pineapple sorbet.
designation: Madeline
points: 88
price: 35.0
province: California
region_1: Napa Valley
region_2: Napa
taster_name: 
taster_twitter_handle: 
title: Prager 2004 Madeline Riesling (Napa Valley)
variety: Riesling
winery: Prager

: 8158
country: US
description: A sweet smell of honeysuckle and grapefruit candy permeates the nose of this bottling, along with cut honeydew melon and apple blossom. Sugary mandarin juice is the primary flavor on the palate, but it's properly offset by acidity a chalky texture.
designation: 
points: 87
price: 29.0
province: California
region_1: Central Coast
region_2: 
taster_name: Matt Kettmann
taster_twitter_handle: @mattkettmann
title: Coquelicot 2016 Riesling
variety: Riesling
winery: Coquelicot

: 12195
country: US
description: Soft and melted in texture, with flavors 

In [27]:
# 파이프라인 중간점검 (요리 풍미 추출 -> 와인 리뷰 검색)
search_wine_review_chain = RunnableLambda(search_wine_review)

chain = dish_flavor_chain | search_wine_review_chain

response = chain.invoke({
    'text': "",
    'image_urls':[
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%252866%2529.png&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832"
    ]
})

print(response)


{'dish_flavor': '허브 스테이크와 시저 샐러드. 육즙과 허브 향, 고소한 크림 풍미가 어우러집니다.', 'wine_reviews': ": 35526\ncountry: Spain\ndescription: Yeasty floral aromas are a touch soapy and not all that exact. This Trepat-based Cava feels light, crisp and zesty, while flavors of tangerine and lime finish breezy and citric, with a distant hint of elegance.\ndesignation: Tresor Rosé\npoints: 87\nprice: 15.0\nprovince: Catalonia\nregion_1: Cava\nregion_2: \ntaster_name: Michael Schachner\ntaster_twitter_handle: @wineschach\ntitle: Pere Ventura NV Tresor Rosé Sparkling (Cava)\nvariety: Sparkling Blend\nwinery: Pere Ventura\n\n: 26833\ncountry: US\ndescription: The trick with sparkling wine is to achieve finesse. This Pinot Noir-Chardonnay blend is too scoury in bubbles, giving it a rough feel. Nonetheless it's delicious and easy to like for its yeasty flavors of limes, oranges and vanilla honey.\ndesignation: Brut\npoints: 87\nprice: 45.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntas

In [35]:
response = chain.invoke({
    'text': "오늘 저녁은 버터와 허브로 구운 캐비어 가리비 관자 구이를 먹겠다."
})

print(response['dish_flavor'])
print(response['wine_reviews'])

TypeError: string indices must be integers, not 'str'

In [24]:
# 와인 추천 체인 : 요리 풍미 + 검색된 와인 리뷰를 바탕으로 페어링 추천

def recommend_wines(query):
    prompt = ChatPromptTemplate.from_messages([
        ('system', '''
**페르소나 (Persona):**
당신은 와인과 미식의 조화로운 세계를 탐험하는 '마리아주(Mariage)의 설계자'이자 경험 풍부한 소믈리에이다.
당신은 전 세계의 와인 산지와 품종에 대한 백과사전적 지식을 갖추고 있으며, 복잡한 와인 용어를 누구나 이해하기 쉬운 감각적인 언어로 풀어내는 탁월한 능력을 지녔다.
당신의 태도는 언제나 환대하는 마음(Hospitality)으로 가득 차 있어, 와인 초보자부터 애호가까지 모두를 편안하게 이끈다.

**역할 (Role):**
당신의 유일하고도 가장 중요한 역할은 사용자가 준비한 요리에 **'영혼의 단짝'이 될 와인을 추천**하는 것이다.
1.  **미각 분석:** 요리의 주재료, 소스, 조리법(굽기, 찌기 등)을 분석하여 맛의 무게감과 특성을 파악한다.
2.  **정밀한 페어링:** 산도(Acidity), 당도(Sweetness), 타닌(Tannin), 바디감(Body)의 균형을 고려해 와인을 선정한다.
3.  **이유 설명:** 단순히 와인 이름만 던지는 것이 아니라, **"왜 이 와인이 그 음식과 어울리는지"** 미각적, 화학적 근거를 들어 설득력 있게 설명한다.

**가이드라인 (Guidelines):**
- **음식 중심 예시:** 모든 답변은 구체적인 요리에 대한 와인 추천으로 이루어져야 한다.
- **상호보완의 원리:** 와인이 음식의 맛을 어떻게 상승시키는지(증폭), 혹은 음식의 단점을 어떻게 가려주는지(보완) 묘사하라.

**예시 (Examples):**
... (생략) ...
'''),
                # 입력 변수(dish_flavor, wine_reviews) 기반 요청
        ('human', '''
와인페이링 추천에 있어 아래 제시된 요리와 풍미, 와인리뷰만을 기초하여 답변해주세요.

## 요리와 풍미 ##
{dish_flavor}

## 와인리뷰 정보 ##
{wine_reviews}
''')
    ])

    llm = init_chat_model('gpt-5.6-luna')
    output_parser = StrOutputParser()

    chain = prompt | llm | output_parser

    return chain    # 체인 객체 반환

recommend_wines_chain = RunnableLambda(recommend_wines)

response = recommend_wines_chain.invoke({
    'dish_flavor': '허브 스테이크는 육즙과 훈연 향이 진하고, 시저 샐러드는 고소·상큼하며 아삭하다.', 
    'wine_reviews': ": 26833\ncountry: US\ndescription: The trick with sparkling wine is to achieve finesse. This Pinot Noir-Chardonnay blend is too scoury in bubbles, giving it a rough feel. Nonetheless it's delicious and easy to like for its yeasty flavors of limes, oranges and vanilla honey.\ndesignation: Brut\npoints: 87\nprice: 45.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: \ntaster_twitter_handle: \ntitle: Kessler-Haak 2012 Brut Sparkling (Sta. Rita Hills)\nvariety: Sparkling Blend\nwinery: Kessler-Haak\n\n: 5664\ncountry: US\ndescription: The winery's annual stainless-steel bottling, this is showy rather than reserved, with ripe Asian pear, tropical flowers, sweet apple blossoms, pineapple and a bit of vanilla on the nose. Nicely fresh, it's bright on the palate, with a sizzle of acidity and grippy chalkiness that frame the poached-pear palate.\ndesignation: Steel\npoints: 90\nprice: 30.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: Matt Kettmann\ntaster_twitter_handle: @mattkettmann\ntitle: Foley 2014 Steel Chardonnay (Sta. Rita Hills)\nvariety: Chardonnay\nwinery: Foley\n\n: 25145\ncountry: US\ndescription: A pink-tinged copper color and a pleasing blend of flavors make this sparkling wine from Schramsberg easy to enjoy. The aromas suggest cherries and cinnamon, the flavor is like tart raspberry and the texture is smooth with fine bubbles.\ndesignation: Brut Rose\npoints: 88\nprice: 28.0\nprovince: California\nregion_1: California\nregion_2: California Other\ntaster_name: Jim Gordon\ntaster_twitter_handle: @gordone_cellars\ntitle: Mirabelle NV Brut Rose Sparkling (California)\nvariety: Sparkling Blend\nwinery: Mirabelle\n\n: 8400\ncountry: US\ndescription: Hay yellow in color, with a wispy shade of pink in the core, this bubbly shows aromas of lemon chiffon, sourdough toast, white peach and a touch of quinine on the nose. Lemon peel and curd show strongly on the palate, laid across a salted cracker flavor—quite delicious and refreshing at once.\ndesignation: Cork Jumper Brut Rosé\npoints: 92\nprice: 42.0\nprovince: California\nregion_1: Santa Maria Valley\nregion_2: Central Coast\ntaster_name: Matt Kettmann\ntaster_twitter_handle: @mattkettmann\ntitle: Riverbench 2014 Cork Jumper Brut Rosé Sparkling (Santa Maria Valley)\nvariety: Sparkling Blend\nwinery: Riverbench\n\n: 32737\ncountry: US\ndescription: This esteemed winery's annual bubbly is crafted much like its still Chardonnays, showing focused aromas of chalk, lemon zest and a telltale brie cheese rind dairy element. It's very yeasty and slightly sour on the mouthwateringly sharp palate, with squeezed limes, lemon pith and underripe kumquat flavors.\ndesignation: 3-D Sparkling\npoints: 90\nprice: 68.0\nprovince: California\nregion_1: Sta. Rita Hills\nregion_2: Central Coast\ntaster_name: Matt Kettmann\ntaster_twitter_handle: @mattkettmann\ntitle: Brewer-Clifton 2012 3-D Sparkling Chardonnay (Sta. Rita Hills)\nvariety: Chardonnay\nwinery: Brewer-Clifton"
})

print(response)


## 추천: Riverbench 2014 **Cork Jumper Brut Rosé Sparkling**

허브 스테이크와 시저 샐러드를 함께 먹는다면, **Riverbench 2014 Cork Jumper Brut Rosé**가 가장 잘 맞습니다.

- **허브 스테이크와의 조화:** 타르트 라즈베리 계열의 산뜻한 과실감이 스테이크의 진한 육즙과 훈연 향에 상쾌한 대비를 만듭니다. 로제의 붉은 과실 풍미가 고기와 자연스럽게 연결되면서도, 무겁게 덮지 않습니다.
- **시저 샐러드와의 조화:** 레몬 껍질과 레몬 커드의 선명한 산도가 시저 드레싱의 고소하고 짭짤한 풍미를 깔끔하게 정리합니다. 샐러드의 아삭한 식감과도 잘 어울리는 상쾌한 인상을 줍니다.
- **전체적인 균형:** 소금기 있는 크래커 풍미와 사워도우 토스트 뉘앙스가 스테이크의 구운 향과 연결되고, 산도가 육즙과 드레싱의 기름기를 씻어내어 다음 한입을 가볍게 만들어줍니다.

### 함께 고려할 선택지
- **Brewer-Clifton 2012 3-D Sparkling Chardonnay:** 레몬·라임의 날카로운 산도와 브리 치즈 껍질 같은 풍미가 시저 샐러드에는 훌륭하지만, 허브 스테이크에는 다소 직선적이고 날카롭게 느껴질 수 있습니다.
- **Mirabelle NV Brut Rosé:** 체리와 라즈베리 풍미가 스테이크와 잘 연결되지만, Riverbench보다 레몬과 짭짤한 크래커의 대비가 덜해 시저 샐러드까지 아우르는 힘은 조금 약합니다.
- **Kessler-Haak 2012 Brut:** 라임·오렌지·바닐라 허니 풍미는 매력적이지만 거친 기포감이 있어, 육즙과 드레싱의 부드러움을 섬세하게 정리하는 데는 덜 적합합니다.

**결론적으로, 진한 스테이크의 풍미를 살리면서 시저 샐러드의 고소함과 상큼함까지 함께 정돈하는 선택은 Riverbench Brut Rosé입니다.**


## 통합 chain

In [29]:
# 요리 풍미
dish_flavor_chain = RunnableLambda(describe_dish_flavor) 
# 풍미 텍스트 -> 유사한 와인 리뷰 검색
search_wine_review_chain = RunnableLambda(search_wine_review)
# 최종 와인 페어링 추천
recommend_wines_chain = RunnableLambda(recommend_wines)


chain = dish_flavor_chain | search_wine_review_chain | recommend_wines_chain

response = chain.invoke({
    'text':"",
    'image_urls' :[
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMTNfMjM4%2FMDAxNzYwMzYzODc4NDk0.me3_kw-eBIiNd75S0W7XdkBiVbUn78pRz5cVlQWV0EEg.VQcftg1p19qRbhxcvaT9Rk80wha9g1VD0f5yVaIe7cUg.PNG%2FImage_fx_%252866%2529.png&type=sc960_832",
        "https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAxODExMjhfMjEy%2FMDAxNTQzNDE1MzEzOTQw.8BT4PzdMbbRctmFFAkLWz3p3G0KywePJxA70TlwTQjcg.8FTmR-SX6Wt7ey27RbV78uzAD8AqedJJKbaAdduuI3Qg.JPEG.story77616%2F20181128_121945.jpg&type=sc960_832"
    ]
})

print(response)

## 1순위: Laetitia 2008 Cuvée M Sparkling  
**92점 · $35 · California, Arroyo Grande Valley**

허브 스테이크와 가장 균형 있게 맞을 와인입니다.

- 스테이크의 **육즙과 풍부한 풍미**에는 와인의 **버터 바른 토스트, 이스트, 감칠맛 나는 savory한 풍미**가 자연스럽게 호응합니다. 와인이 샐러드에만 묻히지 않고 고기의 고소한 맛까지 받아줄 수 있습니다.
- 시저 샐러드의 크리미한 질감에는 리뷰에서 언급된 **부드럽고 크리미한 입안 감촉**이 잘 이어집니다.
- **귤과 라임의 산뜻함**은 샐러드의 상큼함을 증폭하고, 스테이크의 기름진 여운을 산뜻하게 정리합니다.
- 은은한 **꿀 같은 뉘앙스**는 허브의 향과도 부드럽게 연결되어, 요리 전체를 둥글게 묶어줍니다.

**한 줄 평:** 육즙 있는 스테이크를 감당할 풍미와, 시저 샐러드를 상쾌하게 씻어낼 산뜻함을 동시에 갖춘 가장 완성도 높은 선택입니다.

---

## 2순위: Cave de Cleebourg NV Clérotstein Pinot Gris  
**90점 · $22 · Crémant d’Alsace**

조금 더 **가볍고 상큼한 방향**을 원한다면 좋은 선택입니다.

- **레몬과 배, 크리미한 무스**는 시저 샐러드의 고소하고 상큼한 조화와 잘 맞습니다.
- 리뷰의 표현처럼 **풍미는 충분히 풍부하지만 가볍고 드라이하며 상쾌한 스타일**이라, 샐러드의 신선함을 살려줍니다.
- 특히 마무리의 **레몬 같은 산뜻한 여운**이 스테이크의 육즙 뒤에 남는 느끼함을 깔끔하게 보완합니다.

다만 Laetitia보다 풍미가 더 가볍게 느껴질 수 있어, **스테이크보다 시저 샐러드의 비중이 큰 식사**에 더 잘 어울립니다.

---

## 3순위: Schramsberg NV Mirabelle Brut Sparkling  
**87점 · $23 · California, North Coast**

스테이크의 풍부함에 맞춰 조금 더 과실

In [30]:
response = chain.invoke({
    'text':"이따 피자스쿨 베이컨포테이토 피자 먹을거다. 와인은 뭘 먹으면 좋을까?",
    'image_urls' :[
       'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyMTExMjZfOTAg%2FMDAxNjM3ODc3MTI3MDY4.GrivVlwn1aHP8ydQZIA7mym_2AUzq7UaB8zqoTc1d8Mg.z6fe2yK3Tq9E95fHOpHJBq-c3fRUUTGPemiSY939QBcg.JPEG.kkuljo%2F20200406_210410.jpg&type=sc960_832'
    ]
})

print(response)

제시된 와인 중 **베이컨포테이토 피자에는 Robert Biale 2015 Black Chicken Zinfandel**을 가장 추천합니다.

### 1순위: Robert Biale 2015 Black Chicken Zinfandel
- **피자의 짭짤한 베이컨 풍미**를 블루베리와 허클베리의 풍성한 과실감이 부드럽게 감싸줍니다.
- 리뷰에 언급된 **juicy한 흐름과 산도**가 감자와 베이컨의 기름기를 정리해, 한입 뒤 입안을 산뜻하게 만들어줍니다.
- 타닌이 “soft and supple”하므로 피자의 치즈와 충돌하기보다 매끄럽게 이어집니다.
- 베이컨의 훈연향을 압도하기보다는, 풍부한 과실과 산도가 짠맛과 기름기를 보완하는 조합입니다.

### 2순위: Kunde 2010 Kinneybrook Vineyard Chardonnay
버터 팝콘, 버터 토스트, 캐러멜 크림 풍미가 있어 **감자·치즈·베이컨의 고소하고 구운 맛**과 향의 결은 잘 맞습니다. 다만 리뷰상 “달콤하고 오키한” 인상이 강해 피자의 짠맛과 지방감을 더 무겁게 느끼게 할 수 있습니다. 산뜻함보다는 풍미의 일치를 원할 때 적합합니다.

### 덜 추천하는 와인
- **Rivetto 2010 Barbaresco**: 아직 타닌이 단단해 치즈와 베이컨의 짠맛을 거칠게 만들 가능성이 있습니다.
- **Hearst Ranch 2013 Cabernet Sauvignon**: 훈연과 검은 과실은 어울릴 수 있지만, 조밀한 질감과 담배·재 풍미가 피자보다 무겁게 느껴질 수 있습니다.
- **Robert Biale 2013 Black Chicken Zinfandel**: 부드러운 자두와 후추 풍미는 괜찮지만, 2015년 빈티지에 명시된 산도감이 더 분명해 2015년이 피자와의 균형에는 우세합니다.

단, 요리에 제시된 이상적인 방향인 **산뜻한 소비뇽 블랑이나 스파클링 와인**은 이번 목록에 없습니다. 따라서 목록 안에서는 **산도와 부드러운 과실을 갖춘 2015 Black Chicken Zinfand

In [32]:
response = chain.invoke({
    'text':"섞어마시기 좋은 와인"
})

print(response)

요리 사진을 다시 올려 주세요. 확인 후 어울리는 와인과 믹스 조합을 50자 이내 추천하겠습니다.


In [33]:
response = chain.invoke({
    'text':"고추잡채",
    'image_urls' :[
       'https://search.pstatic.net/common/?src=http%3A%2F%2Fblogfiles.naver.net%2FMjAyNTEwMDNfNDYg%2FMDAxNzU5NDkwODcxMjE4.nDH5zOdTi5HvhLUtgYXtl9ps1jQLWn5MNC1XNrChF8Yg.xADP7ofugONTY3vwbnFeS7cQQmPTBXQfjC87WNsFF3Ig.JPEG%2FIMG_3801.JPG&type=sc960_832'
    ]
})

print(response)

## 추천 와인: **Viña Requingua 2014 Los Riscos Pinot Grigio**

고추잡채에는 제시된 와인 중 **칠레산 피노 그리지오**가 가장 무난한 선택입니다.

- 고추잡채의 **매콤함과 돼지고기의 고소함**에는 타닌이 강한 레드보다, 리뷰상 타닌감이 드러나지 않는 피노 그리지오가 충돌이 적습니다.
- 이 와인은 풍미가 **담백하고 평평하며 바나나 느낌**을 보인다고 평가되어, 음식의 매콤달콤한 양념을 거칠게 증폭시키기보다 비교적 조용히 받쳐줄 수 있습니다.
- 특히 Aglianico처럼 **쓴맛·산도·타닌이 풍부한 와인**은 고추의 매운맛과 만나 쓴맛과 떫은 느낌을 키울 수 있어 피하는 편이 좋습니다.

다만 이 피노 그리지오는 리뷰에서 **생동감과 선명한 마무리가 부족하다**고 평가된 만큼, 고추잡채의 아삭함을 산뜻하게 끌어올리는 이상적인 페어링은 아닙니다. 제시된 선택지 안에서 매운 양념과 가장 크게 충돌하지 않는 와인이라는 점에서 추천합니다.

**서빙 팁:** 충분히 차갑게 마시면 와인의 둔한 인상이 조금 줄고, 고추잡채의 매콤함도 한결 부드럽게 느껴질 것입니다.
